In [ ]:
%env JAX_PLATFORMS=cpu
%env JAX_ENABLE_X64=True

In [ ]:
import sys
from pathlib import Path

cwd = Path.cwd()
root = next(
    (path for path in (cwd, *cwd.parents) if (path / "experiments/test_bench/bench.py").is_file()),
    None,
)
if root is None:
    raise RuntimeError("Open this notebook from the repository or its experiment folder.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from experiments.test_bench.bench import (
    LossFigure,
    build_task,
    build_task_network,
    evaluate,
    plot_rollout,
)
from spikegd.test_bench.network import load_net
from spikegd.test_bench.rules import rflo
from spikegd.test_bench.training import Trainer, TrainerConfig

import matplotlib.pyplot as plt
import numpy as np

In [ ]:
plt.style.use("spikegd.utils.plotstyle")

# Run experiments

Train an event-driven QIF network with an online RFLO rule. The default cyclic example runs without a dataset. Run the cells from a fresh kernel in an environment where the repository is installed. See the [README](README.md) for setup and dataset paths.

## Configuration

Choose `cyclic`, `copy`, `add`, `mnist`, or `timit`. Pass a local dataset path in `task_options` for MNIST or TIMIT. The online Jacobian options are `murray`, `e`, `exact_decay`, and `exact_own`.

In [ ]:
task_name = "cyclic"
task_options = {}
# task_options = {"data_root": "/path/to/dataset"}

jacobian = "murray"
signal = "W_out_targets"  # Or "B" for random feedback, or "targets_only".
config = TrainerConfig(n_trials=10, checkpoints_across_trials=10, K="auto")

evaluation_split = "train"  # Use "test" for tasks with a held-out split.
export_path = None  # For example, root / "experiments/test_bench/output/network.npz".

## Train and evaluate

Training records the loss at the configured trials. The rollout below uses the final weights. Set `evaluation_split = "test"` to inspect a held-out sample for tasks with that split.

In [ ]:
task = build_task(task_name, **task_options)
net = build_task_network(task)
rule = rflo(jacobian, signal=signal)
trainer = Trainer(net, task, rule, config)
task.describe()
net.describe()

record = trainer.train()

sample, y, events, final_loss = evaluate(trainer, record, index=0, split=evaluation_split)
print(f"Final loss on {evaluation_split} sample 0: {final_loss:.6g}")

# Plot results

## Training loss

In [ ]:
LossFigure().draw(record)
plt.show()

## Final readout and spike raster

In [ ]:
plot_rollout(task, sample, y, events)
plt.show()

# Export results

Set `export_path` to save the network structure and final parameters as an NPZ archive. The cell reloads the archive and checks the parameters and readout.

In [ ]:
if export_path is not None:
    export_path = Path(export_path)
    export_path.parent.mkdir(parents=True, exist_ok=True)
    net.export(export_path, record.params)
    restored_net, restored_params = load_net(export_path)
    for name, value in record.params.items():
        np.testing.assert_allclose(restored_params[name], value)
    restored_y, _ = restored_net.forward(
        restored_params, sample.spikes_in, sample.T, trainer.K, read_at=sample.read_at,
    )
    np.testing.assert_allclose(restored_y, y)
    print(f"Verified {export_path}")